In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from viz_style import apply_style, FS_ANNOT_LG, FS_ANNOT_SM
apply_style()

import MixedEffectsModeling.config as config
from MixedEffectsModeling.validation import lobo_mmd

LOBO = config.LOBO_MIXED_DIR
FIG = LOBO / 'Figures'
FIG.mkdir(parents=True, exist_ok=True)

In [ ]:
from matplotlib.patches import Patch


def _p_to_asterisk(p):
    if p < 0.001:
        return '***'
    if p < 0.01:
        return '**'
    if p < 0.05:
        return '*'
    return ''


def _short_batch_label(batch):
    name, _, num = batch.replace(' et al.', '').partition('_Batch_')
    return f'{name}_{num}' if num else name


def plot_mmd_bar(df, raw, out_path):
    # Violin of the permutation null (what MMD^2 looks like under no batch/disease
    # difference) with the observed statistic marked -- a bar+asterisk collapses this
    # whole null distribution into one number, hiding how far out the observation sits.
    d = df.sort_values('mmd2', ascending=False).reset_index(drop=True)
    labels = [_short_batch_label(b) for b in d['batch']]
    x = np.arange(len(d))
    fig, ax = plt.subplots(figsize=(4, 6))
    for xi, b in zip(x, d['batch']):
        r = raw[b]
        vp = ax.violinplot(r['mmd2_null'], positions=[xi], widths=0.7, showmeans=False, showextrema=False)
        for body in vp['bodies']:
            body.set_facecolor('#c8cdd2')
            body.set_edgecolor('none')
            body.set_alpha(0.7)
        ax.scatter([xi], [r['mmd2_obs']], color='#00C78B', zorder=3, s=60, edgecolors='black', linewidth=0.6)
    y_span = d['mmd2'].max() - d['mmd2'].min()
    for xi, b, p in zip(x, d['batch'], d['perm_p']):
        label = _p_to_asterisk(p)
        if label:
            ax.text(xi, raw[b]['mmd2_obs'] + 0.03 * y_span, label, ha='center', va='bottom')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('MMD$^2$ (permutation null vs. observed)')
    ax.set_xlabel('LOBO batch')
    ax.legend(handles=[Patch(facecolor='#c8cdd2', label='permutation null'),
                       Patch(facecolor='#00C78B', label='observed')],
             frameon=False, loc='upper right')
    plt.tight_layout()
    fig.savefig(out_path, bbox_inches='tight')
    return fig


def plot_mmd_direction(df, raw, out_path, p_col='p_direction'):
    from scipy.stats import gaussian_kde
    d = df.sort_values('mmd2', ascending=False).reset_index(drop=True)
    n_batches = len(d)
    
    fig, axes = plt.subplots(n_batches, 1, figsize=(7, 1.4 * n_batches), sharex=True)
    if n_batches == 1:
        axes = [axes]
        
    color_hc = '#A4AFB8'
    color_dis = '#00C78B'
    
    for ax, (_, row) in zip(axes, d.iterrows()):
        b = row['batch']
        r = raw[b]
        
        d_hc = np.asarray(r['d_hc'])
        d_dis = np.asarray(r['d_dis'])
        
        x_min = min(d_hc.min(), d_dis.min())
        x_max = max(d_hc.max(), d_dis.max())
        x_margin = (x_max - x_min) * 0.2
        x_grid = np.linspace(x_min - x_margin, x_max + x_margin, 300)
        
        kde_hc = gaussian_kde(d_hc)(x_grid)
        kde_dis = gaussian_kde(d_dis)(x_grid)
        
        ax.plot(x_grid, kde_hc, color=color_hc, lw=1.5)
        ax.fill_between(x_grid, kde_hc, color=color_hc, alpha=0.35)
        
        ax.plot(x_grid, kde_dis, color=color_dis, lw=1.5)
        ax.fill_between(x_grid, kde_dis, color=color_dis, alpha=0.35)
        
        mean_hc = d_hc.mean()
        mean_dis = d_dis.mean()
        max_y = max(kde_hc.max(), kde_dis.max())
        
        ax.axvline(mean_hc, color=color_hc, linestyle='--', lw=1.5, zorder=3)
        ax.axvline(mean_dis, color=color_dis, linestyle='--', lw=1.5, zorder=3)
        
        y_bar = max_y * 1.15
        ax.annotate('', xy=(mean_hc, y_bar), xytext=(mean_dis, y_bar),
                    arrowprops=dict(arrowstyle='<->', color='black', lw=1.2))
        
        p_val = row.get(p_col, 1.0)
        asterisk = _p_to_asterisk(p_val)
        delta = abs(mean_dis - mean_hc)
        
        mid_x = (mean_hc + mean_dis) / 2
        text_str = f"Δ={delta:.3f} ({asterisk})"
        
        ax.text(mid_x, y_bar + max_y * 0.08, text_str,
                ha='center', va='bottom', fontweight='bold', color='black')
        
        label = _short_batch_label(b) if '_short_batch_label' in globals() else str(b)
        ax.set_ylabel(label, rotation=0, ha='right', va='center')
        ax.set_ylim(0, max_y * 1.55)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, axis='x', linestyle=':', alpha=0.4)
        
    axes[-1].set_xlabel('Kernel embedding distance from HC reference')
    
    legend_elements = [
        Patch(facecolor=color_hc, edgecolor=color_hc, alpha=0.5, label='Held-out HC'),
        Patch(facecolor=color_dis, edgecolor=color_dis, alpha=0.5, label='Disease')
    ]
    axes[0].legend(handles=legend_elements, loc='upper right', frameon=False)
    
    plt.tight_layout()
    fig.savefig(out_path, bbox_inches='tight', dpi=300)
    return fig

In [ ]:
# 0. Raw MMD summary (held-out HC vs held-out disease, per Tier-A batch with n_hc >= lobo_mmd.MIN_N_HC)
# no SHASH correction -- Z_test.npy / cv_zscores.pkl are used exactly as scored
df_raw = lobo_mmd.mmd_summary_cached(force=False, shash=False)
raw_raw = lobo_mmd.mmd_raw_cached(force=False, shash=False)
print(df_raw.to_string(index=False))

n_sig_raw = int(((df_raw['perm_p'] < 0.05) & df_raw['disease_farther']).sum())
print(f"\n{n_sig_raw}/{len(df_raw)} batches: MMD significant (p<0.05) AND disease farther "
     f"from the in-fold HC reference than the held-out-HC noise floor. (raw, no SHASH)")

In [ ]:
# 1. Raw MMD plots
plot_mmd_bar(df_raw, raw_raw, FIG / 'mmd_bar.png')
plot_mmd_direction(df_raw, raw_raw, FIG / 'mmd_direction.png')

In [ ]:
# 2. SHASH-corrected MMD summary -- per-batch SHASH fit on that batch's own train-fold
# in-sample Z (lobo_engine.run_one_batch), applied to the held-out LOBO Z before comparing
df_shash = lobo_mmd.mmd_summary_cached(force=False, shash=True)
raw_shash = lobo_mmd.mmd_raw_cached(force=False, shash=True)
print(df_shash.to_string(index=False))

n_sig_shash = int(((df_shash['perm_p'] < 0.05) & df_shash['disease_farther']).sum())
print(f"\n{n_sig_shash}/{len(df_shash)} batches: MMD significant (p<0.05) AND disease farther "
     f"from the in-fold HC reference than the held-out-HC noise floor. (SHASH-corrected)")

In [ ]:
# 3. SHASH-corrected MMD plots
plot_mmd_bar(df_shash, raw_shash, FIG / 'mmd_bar_shash.png')
plot_mmd_direction(df_shash, raw_shash, FIG / 'mmd_direction_shash.png')

In [ ]:
# 4. Raw vs SHASH-corrected, side by side -- does the raw-scale call survive de-skewing?
cmp = df_raw.merge(df_shash, on='batch', suffixes=('_raw', '_shash'))
cmp['sig_raw'] = (cmp['perm_p_raw'] < 0.05) & cmp['disease_farther_raw']
cmp['sig_shash'] = (cmp['perm_p_shash'] < 0.05) & cmp['disease_farther_shash']
print(cmp[['batch', 'mmd2_raw', 'perm_p_raw', 'sig_raw', 'mmd2_shash', 'perm_p_shash', 'sig_shash']].to_string(index=False))
print(f"\nagree (both sig or both not): {(cmp['sig_raw'] == cmp['sig_shash']).sum()}/{len(cmp)}")

In [ ]:
from MixedEffectsModeling.validation.lobo_engine import compute_ood

compute_ood()

In [ ]:
# OOD-filtered MMD, raw and SHASH-corrected
df_ood = lobo_mmd.mmd_summary_cached(force=False, shash=False, ood_filter=True)
raw_ood = lobo_mmd.mmd_raw_cached(force=False, shash=False, ood_filter=True)
df_shash_ood = lobo_mmd.mmd_summary_cached(force=False, shash=True, ood_filter=True)
raw_shash_ood = lobo_mmd.mmd_raw_cached(force=False, shash=True, ood_filter=True)

for label, d in [('ood', df_ood), ('shash+ood', df_shash_ood)]:
    n_sig = int(((d['perm_p'] < 0.05) & d['disease_farther']).sum())
    print(f"[{label}] {n_sig}/{len(d)} batches significant (p<0.05) AND disease farther")
print()
print(df_shash_ood.to_string(index=False))

In [ ]:
# shash+ood plots (the most rigorous variant: batch-refit + SHASH-decorrelated + extrapolation-flagged)
plot_mmd_bar(df_shash_ood, raw_shash_ood, FIG / 'mmd_bar_shash_ood.png')
plot_mmd_direction(df_shash_ood, raw_shash_ood, FIG / 'mmd_direction_shash_ood.png')

In [ ]:
# All 4 variants side by side -- does the raw-scale call survive de-skewing AND extrapolation-flagging?
variants = {'raw': df_raw, 'shash': df_shash, 'ood': df_ood, 'shash_ood': df_shash_ood}
sig = pd.DataFrame({name: (d.set_index('batch')['perm_p'] < 0.05) & d.set_index('batch')['disease_farther']
                    for name, d in variants.items()})
mmd2 = pd.DataFrame({name: d.set_index('batch')['mmd2'] for name, d in variants.items()})

print('significant (p<0.05) AND disease farther, per variant:')
print(sig.to_string())
print(f"\nagree across all 4 variants: {(sig.nunique(axis=1) == 1).sum()}/{len(sig)} batches")
print('\nmmd2 by variant:')
print(mmd2.round(4).to_string())

In [ ]:
import json
import pickle

import seaborn as sns
from scipy.stats import mannwhitneyu, norm

from MixedEffectsModeling.core.calibration import bh_fdr_reject

FDR_Q, Z_CAP = 0.05, 10.0
NSIG_PATH = LOBO / 'lobo_nsig.csv'
lobo_meta = sorted(LOBO.glob('*/meta.json'))

if NSIG_PATH.exists():
    nsig = pd.read_csv(NSIG_PATH)
else:
    # Per-batch SHASH already fit (lobo_engine.run_one_batch) on that batch's own
    # train-fold (HC minus the held-out batch) in-sample Z and applied to its held-out
    # Z -- Z_test_shash.npy -- never fit on the held-out Z it's graded against.
    rows = []
    for p in lobo_meta:
        m = json.load(open(p))
        batch = m['batch_id']
        Zb = np.nan_to_num(np.load(p.parent / 'Z_test_shash.npy').astype(np.float32), nan=0.0)
        Zb = np.clip(Zb, -Z_CAP, Z_CAP)
        ns = [int(bh_fdr_reject(r, q=FDR_Q).sum()) for r in 2 * norm.sf(np.abs(Zb))]
        rows += [(batch, nm, bool(hc), v)
                 for nm, hc, v in zip(m['test_names'], m['test_is_hc'], ns)]
    nsig = pd.DataFrame(rows, columns=['batch', 'sample', 'is_hc', 'n_sig'])
    nsig.to_csv(NSIG_PATH, index=False)

rows = []
for b, g in nsig.groupby('batch'):
    h = g.loc[g['is_hc'], 'n_sig'].values
    d = g.loc[~g['is_hc'], 'n_sig'].values
    if len(h) == 0 or len(d) == 0:
        continue
    u, pu = mannwhitneyu(d, h, alternative='greater')
    rows.append(dict(batch=b, n_hc=len(h), n_dis=len(d), hc_med=np.median(h), dis_med=np.median(d),
                     auc=u / (len(d) * len(h)), mw_p=pu))
nsig_stats = pd.DataFrame(rows).sort_values('auc', ascending=False)
nsig_stats.to_csv(LOBO / 'batch_matched_nsig_stats.csv', index=False)

# same test on true nulls: HC vs the rest of its own batch's HC, must stay at or under 0.05
fp = [((np.delete(h, i) >= h[i]).sum() + 1) / len(h)
      for _, g in nsig.groupby('batch')
      for h in [g.loc[g['is_hc'], 'n_sig'].values] if len(h) >= 19
      for i in range(len(h))]

print(f"batches with both HC and disease: {len(nsig_stats)}  "
      f"AUC>0.5: {(nsig_stats['auc'] > 0.5).sum()}  "
      f"Mann-Whitney p<0.05: {(nsig_stats['mw_p'] < 0.05).sum()}")
print(f"HC-vs-HC leave-one-out false-positive rate: {np.mean(np.array(fp) < 0.05):.4f} "
      f"(target 0.05, n={len(fp)})\n")
print(nsig_stats.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

In [ ]:
# 6. Batch-matched null vs observation, and the within-batch effect size
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

order = nsig_stats.sort_values('auc')['batch'].tolist()
plot = nsig[nsig['batch'].isin(order)].copy()
plot['group'] = np.where(plot['is_hc'], 'HC (held-out)', 'Disease')
labels = [_short_batch_label(b) for b in order]

sns.stripplot(data=plot, y='batch', x='n_sig', hue='group', order=order,
              hue_order=['HC (held-out)', 'Disease'], palette=['#8c8c8c', '#2a6099'],
              dodge=True, orient='h', size=3, alpha=0.6, jitter=0.25, ax=axes[0])
axes[0].set_xscale('symlog')
axes[0].set_xlim(left=-0.1)
axes[0].set_yticklabels(labels)
axes[0].set_xlabel('BH-significant genes per sample (q=0.05)')
axes[0].set_ylabel('')
axes[0].set_title('Batch-matched null and observation\n(same batch, same LOBO engine)')
axes[0].legend(frameon=False, fontsize=FS_ANNOT_SM, loc='lower right')

d = nsig_stats.sort_values('auc')
axes[1].hlines(range(len(d)), 0.5, d['auc'], color='#cccccc', lw=1.2, zorder=1)
axes[1].scatter(d['auc'], range(len(d)), s=np.clip(d['n_dis'], 10, 120),
                color=np.where(d['mw_p'] < 0.05, '#2a6099', '#aec6e8'), zorder=3)
axes[1].axvline(0.5, color='#d64545', ls='--', lw=1.2)
axes[1].text(0.5, len(d) - 0.4, ' no signal', color='#d64545', fontsize=FS_ANNOT_SM, ha='left')
axes[1].set_yticks(range(len(d)))
axes[1].set_yticklabels([f"{_short_batch_label(b)} {_p_to_asterisk(p)}"
                         for b, p in zip(d['batch'], d['mw_p'])], fontsize=FS_ANNOT_SM)
axes[1].set_xlabel('AUC: P(disease deviates more than HC, same batch)')
axes[1].set_title('Within-batch effect size\n(point size = n disease; filled = p<0.05)')

plt.tight_layout()
fig.savefig(FIG / 'batch_matched_nsig.png', bbox_inches='tight')